# Feature Table Inspection

モデル構築に使用する施設のデータと、モデルの評価（external evaluation）に使う施設のデータを分割して保存する

## Set Config

In [25]:
# ライブラリのインポート
import os
import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import seaborn as sns

In [26]:
# フォルダ設定
data_dir = '../data/'
output_dir = '../output/'

# date
date = '20260316_emergency_first24h'

# 特徴量tableのファイル名
feature_table = f'{date}_features.csv'

In [27]:
# 保存するファイル名の指定
df_development_file_name = f'{date}_df_development_raw.csv'
# df_evaluation_file_name = f'{date}_df_evaluation_raw.csv'

## Load data

In [28]:
df = pl.read_csv(os.path.join(data_dir + feature_table))

In [29]:
# record数と症例数を確認
print('Number of records:', len(df))
print('Number of icu_stay_id:', len(df['icu_stay_id'].unique()))
print('Average number of records per icu_stay_id:', len(df) / len(df['icu_stay_id'].unique()))

Number of records: 27307
Number of icu_stay_id: 27307
Average number of records per icu_stay_id: 1.0


# Inspect Feature Table

In [30]:
def describe_df(df: pl.DataFrame, start: int, end: int) -> pl.DataFrame:
    return df[:, start:end].describe()

In [31]:
describe_df(df, 0, 10)

statistic,icu_stay_id,time_window_index,start_time,end_time,time_zero,out_time,label_delirium_indication,ph,pao2,pvo2
str,f64,f64,str,str,str,str,f64,f64,f64,f64
"""count""",27307.0,27307.0,"""27307""","""27307""","""27307""","""27307""",27307.0,25085.0,23639.0,4830.0
"""null_count""",0.0,0.0,"""0""","""0""","""0""","""0""",0.0,2222.0,3668.0,22477.0
"""mean""",5.7745e17,0.0,null,null,null,null,0.240891,7.397488,113.605216,45.184741
"""std""",3.3235e17,0.0,null,null,null,null,0.427632,0.072122,51.44951,27.459882
"""min""",2.8023e13,0.0,"""2200-01-01 00:00:00 UTC""","""2200-01-02 00:00:00 UTC""","""2200-01-01 00:00:00 UTC""","""2200-01-02 00:17:00 UTC""",0.0,6.69,14.1,2.7
"""25%""",2.9141e17,0.0,null,null,null,null,0.0,7.3665,82.75,31.7
"""50%""",5.7964e17,0.0,null,null,null,null,0.0,7.406,99.85,40.0
"""75%""",8.6540e17,0.0,null,null,null,null,0.0,7.441,130.0,51.5
"""max""",1.1528e18,0.0,"""2210-05-14 19:22:00 UTC""","""2210-05-15 19:22:00 UTC""","""2210-05-14 19:22:00 UTC""","""2210-05-31 16:26:00 UTC""",1.0,7.716,689.0,560.0


In [32]:
describe_df(df, 10, 20)

statistic,paco2,pvco2,bg_lactate,bg_glucose,gcs_e,gcs_v,gcs_m,body_weight,bt50,hr10
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",23650.0,4836.0,25081.0,23159.0,26832.0,26832.0,26832.0,22434.0,27108.0,27306.0
"""null_count""",3657.0,22471.0,2226.0,4148.0,475.0,475.0,475.0,4873.0,199.0,1.0
"""mean""",38.25441,44.82195,2.182578,149.777085,2.773554,2.882938,4.639926,59.785798,37.108573,74.523141
"""std""",8.404041,13.813289,2.252429,54.549243,1.239673,1.902729,1.943419,15.400863,1.072701,16.344128
"""min""",5.3,12.4,0.2,3.0,1.0,1.0,1.0,16.0,10.4,0.0
"""25%""",33.65,37.4,1.0,115.0,1.0,1.0,4.0,49.1,36.6,63.0
"""50%""",37.6,42.7,1.5,138.0,3.0,2.0,6.0,58.25,37.1,73.0
"""75%""",41.7,49.0,2.4,169.0,4.0,5.0,6.0,68.4,37.7,84.0
"""max""",255.45,197.0,36.0,1017.0,4.0,5.0,6.0,200.0,43.0,149.0


In [33]:
describe_df(df, 20, 30)

statistic,hr50,hr90,hr_sd,rr10,rr50,rr90,rr_sd,sbp10,sbp50,sbp90
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",27306.0,27306.0,27306.0,27293.0,27293.0,27293.0,27289.0,27305.0,27305.0,27305.0
"""null_count""",1.0,1.0,1.0,14.0,14.0,14.0,18.0,2.0,2.0,2.0
"""mean""",84.028199,97.790877,9.828564,13.683206,17.08101,22.047069,3.946311,99.768207,118.716133,140.02761
"""std""",17.526753,19.760891,4.952514,5.125908,5.870473,6.70523,1.801904,20.07824,21.400687,24.086412
"""min""",33.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",71.0,84.0,6.439214,12.0,15.0,19.0,2.785173,86.3,104.0,123.6
"""50%""",82.0,96.0,8.764246,14.0,17.0,22.0,3.612854,98.0,117.0,138.0
"""75%""",95.0,110.0,12.073128,16.0,20.0,25.0,4.684966,111.5,131.0,154.4
"""max""",168.0,203.0,51.3337,40.0,53.0,68.0,19.822418,210.0,239.5,300.0


In [34]:
describe_df(df, 30, 40)

statistic,sbp_sd,mbp10,mbp50,mbp90,mbp_sd,dbp10,dbp50,dbp90,dbp_sd,spo2_10
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",27296.0,27305.0,27305.0,27305.0,27296.0,27305.0,27305.0,27305.0,27296.0,27306.0
"""null_count""",11.0,2.0,2.0,2.0,11.0,2.0,2.0,2.0,11.0,1.0
"""mean""",17.327957,68.536788,80.256357,94.910029,12.524239,51.861751,61.027559,73.599275,10.838375,95.290449
"""std""",6.95765,13.580124,14.474626,17.434679,5.849844,12.040516,13.129709,17.241386,6.07886,4.138034
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",12.53171,60.0,70.0,83.0,8.775048,44.0,52.0,62.0,7.148323,94.0
"""50%""",16.330919,67.0,78.333333,93.0,11.541784,51.0,60.0,71.0,9.53094,96.0
"""75%""",20.985875,76.0,89.0,105.0,15.005235,58.8,69.0,82.0,12.766157,97.0
"""max""",101.229392,210.0,272.0,300.0,107.583817,210.0,222.0,300.0,108.150872,100.0


In [35]:
describe_df(df, 40, 50)

statistic,spo2_50,spo2_90,spo2_sd,bp_invasive,wbc,hemoglobin,platelet,sodium,creatinine,potassium
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",27306.0,27306.0,27305.0,27306.0,26045.0,25931.0,26041.0,25997.0,25989.0,25980.0
"""null_count""",1.0,1.0,2.0,1.0,1262.0,1376.0,1266.0,1310.0,1318.0,1327.0
"""mean""",97.746288,99.172691,2.028008,0.820589,11.190295,11.025421,185.115971,138.945532,1.574929,4.084003
"""std""",2.337318,1.33971,1.627766,0.383703,7.934416,2.301907,94.36857,5.454289,1.861439,0.641883
"""min""",12.0,66.0,0.0,0.0,0.0,0.0,0.0,103.0,0.015,1.3
"""25%""",97.0,99.0,1.242203,1.0,7.2,9.35,121.0,136.0,0.68,3.7
"""50%""",98.0,100.0,1.717909,1.0,9.95,10.9,175.0,139.0,0.925,4.0
"""75%""",100.0,100.0,2.338423,1.0,13.55,12.6,234.0,141.5,1.52,4.4
"""max""",100.0,100.0,38.681021,1.0,419.85,22.1,1732.0,187.0,31.36,9.5


In [36]:
describe_df(df, 50, 60)

statistic,aspartate_aminotransferase,blood_urea_nitrogen,alanine_aminotransferase,total_bilirubin,crp,albumin,hospital_id,icu_admit_time,female,age
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",25675.0,25939.0,25677.0,25598.0,24255.0,24190.0,27307.0,27307.0,27307.0,27307.0
"""null_count""",1632.0,1368.0,1630.0,1709.0,3052.0,3117.0,0.0,0.0,0.0,0.0
"""mean""",150.245628,27.88312,81.111987,1.138287,7.249111,2.908893,4.696818,13.46446,0.366902,65.635881
"""std""",689.397998,23.455255,312.885122,1.80484,8.820947,0.716977,3.580339,6.00504,0.481968,17.371795
"""min""",4.0,1.0,0.5,0.075,0.0,0.1,1.0,0.0,0.0,15.0
"""25%""",21.0,13.5,13.5,0.5,0.91,2.4,1.0,10.0,0.0,55.0
"""50%""",35.0,20.0,22.5,0.75,3.545,2.9,3.0,14.0,0.0,70.0
"""75%""",73.0,33.5,46.0,1.175,10.4005,3.4,7.0,18.0,1.0,78.0
"""max""",41900.0,315.5,7988.0,43.2,61.0345,5.6,11.0,23.0,1.0,99.0


In [37]:
describe_df(df, 60, 70)

statistic,icu_admission_type,diagnosis_category,covid19,sepsis,stroke,poisoning,anaphylaxis,burn,temperature_disorder,hanging_asphyxiation
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""count""","""27307""","""27307""",27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0
"""null_count""","""0""","""0""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",null,null,0.059692,0.028637,0.064782,0.019775,0.001062,0.009741,0.011719,0.001794
"""std""",null,null,0.236919,0.166788,0.246145,0.139229,0.032572,0.098217,0.107618,0.042323
"""min""","""emergency""","""anaphylaxis""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",null,null,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""max""","""emergency""","""trauma""",1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [38]:
describe_df(df, 70, 80)

statistic,trauma,infection,neoplasms,circulatory,respiratory,digestive,neurological,genitourinary,electrolyte_metabolic_disorders,musculoskeletal
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",0.125792,0.097484,0.054418,0.251364,0.045922,0.064306,0.027282,0.021057,0.042517,0.016882
"""std""",0.331621,0.296621,0.226845,0.433805,0.20932,0.245301,0.162908,0.143577,0.201768,0.128832
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
"""max""",1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [39]:
describe_df(df, 80, 90)

statistic,psychiatric,congenital,undefined,other,icd10,apache2_score,charlson_comorbidity_index,myocardial_infarct,congestive_heart_failure,peripheral_arterial_disease
str,f64,f64,f64,f64,str,f64,f64,f64,f64,f64
"""count""",27307.0,27307.0,27307.0,27307.0,"""26895""",27307.0,27307.0,27307.0,27307.0,27307.0
"""null_count""",0.0,0.0,0.0,0.0,"""412""",0.0,0.0,0.0,0.0,0.0
"""mean""",0.009338,0.003479,0.020215,0.022741,null,26.683268,3.471125,0.058227,0.134764,0.086901
"""std""",0.096184,0.058881,0.140736,0.149081,null,9.718215,2.905277,0.234176,0.341478,0.281695
"""min""",0.0,0.0,0.0,0.0,"""<TBD>""",2.0,0.0,0.0,0.0,0.0
"""25%""",0.0,0.0,0.0,0.0,null,20.0,2.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,null,26.0,3.0,0.0,0.0,0.0
"""75%""",0.0,0.0,0.0,0.0,null,33.0,4.0,0.0,0.0,0.0
"""max""",1.0,1.0,1.0,1.0,"""Z988""",61.0,25.0,1.0,1.0,1.0


In [40]:
describe_df(df, 90, 100)

statistic,cerebrovascular_disease,dementia,chronic_pulmonary_disease,connective_tissue_disease,peptic_ulcer_disease,mild_liver_disease,diabetes_without_cc,diabetes_with_cc,paraplegia,renal_disease
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",0.113011,0.012854,0.04281,0.031311,0.11671,0.058923,0.043688,0.024243,0.004871,0.06112
"""std""",0.316612,0.112646,0.202431,0.174159,0.32108,0.235484,0.204405,0.153805,0.06962,0.239554
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""max""",1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [41]:
describe_df(df, 100, 110)

statistic,malignant_cancer,severe_liver_disease,metastatic_solid_tumor,aids,age_score,fio2_ordered,peep_ordered,inspiratory_pressure,is_control_ventilation,time_since_last_control_ventilation
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",27307.0,27307.0,27307.0,27307.0,27307.0,13539.0,13636.0,11329.0,15773.0,15773.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,13768.0,13671.0,15978.0,11534.0,11534.0
"""mean""",0.121581,0.011975,0.027063,0.008276,2.160691,60.4117,NaN,NaN,1.0,0.0
"""std""",0.326807,0.108775,0.162269,0.090598,1.389275,25.558583,NaN,NaN,0.0,0.0
"""min""",0.0,0.0,0.0,0.0,0.0,20.0,0.0,-5.0,1.0,0.0
"""25%""",0.0,0.0,0.0,0.0,1.0,40.0,5.0,12.0,1.0,0.0
"""50%""",0.0,0.0,0.0,0.0,2.0,50.0,8.0,15.0,1.0,0.0
"""75%""",0.0,0.0,0.0,0.0,3.0,85.0,10.0,21.0,1.0,0.0
"""max""",1.0,1.0,1.0,1.0,4.0,100.0,450.0,500.0,1.0,0.0


In [42]:
describe_df(df, 110, 120)

statistic,mv_duration,rass_score,respiration_24hours,coagulation_24hours,liver_24hours,cardiovascular_24hours,cns_24hours,renal_24hours,sofa_24hours,input_amount
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",15773.0,23898.0,23659.0,26954.0,26780.0,27305.0,27135.0,27057.0,27307.0,27307.0
"""null_count""",11534.0,3409.0,3648.0,353.0,527.0,2.0,172.0,250.0,0.0,0.0
"""mean""",1146.19172,-2.00226,1.603449,0.619871,0.369249,1.616371,1.166925,1.301881,6.335555,145.51852
"""std""",396.285933,1.947918,1.18572,0.95011,0.722555,1.374505,1.437408,1.599987,3.937942,307.445419
"""min""",0.0,-5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",1002.0,-4.0,1.0,0.0,0.0,1.0,0.0,0.0,3.0,1.0
"""50%""",1330.0,-1.0,2.0,0.0,0.0,1.0,1.0,0.0,6.0,30.530973
"""75%""",1427.0,0.0,2.0,1.0,0.5,3.0,2.0,3.0,9.0,137.0
"""max""",3714.0,3.0,4.0,4.0,4.0,4.0,4.0,4.0,23.5,5207.333333


In [43]:
describe_df(df, 120, 130)

statistic,active_ingredient_propofol,active_ingredient_noradrenaline,active_ingredient_fentanyl,active_ingredient_dexmedetomidine,active_ingredient_dopamine,active_ingredient_midazolam,active_ingredient_dobutamine,active_ingredient_landiolol,active_ingredient_vasopressin,active_ingredient_remifentanil
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",544.710578,2.508905,0.346088,0.392167,23.813112,16.577438,19.33301,5.154197,1.645598,0.016626
"""std""",1101.823842,6.251585,0.48091,1.864178,107.699554,52.616262,77.187442,36.565499,7.881884,0.180779
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""75%""",588.666667,1.634,0.61,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""max""",10242.0,122.121,11.0,33.6,2138.5,918.833333,1441.77,1717.5,264.787745,5.61525


In [44]:
describe_df(df, 130, 140)

statistic,active_ingredient_thiamylal,active_ingredient_adrenaline,active_ingredient_vecuronium,active_ingredient_rocuronium,window_total_output,transfusion_rbc,transfusion_platelet,transfusion_plasma,transfusion_albumin,fio2_ordered_is_current_value
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""count""",27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0,27307.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",3.683812,0.138573,0.734645,23.507674,1872.940523,137.178512,17.378597,110.450378,42.720576,0.495807
"""std""",108.308369,1.423869,5.187524,87.928627,1589.664218,451.720849,100.702085,453.571918,175.064014,0.499992
"""min""",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""25%""",0.0,0.0,0.0,0.0,935.714,0.0,0.0,0.0,0.0,0.0
"""50%""",0.0,0.0,0.0,0.0,1565.742,0.0,0.0,0.0,0.0,0.0
"""75%""",0.0,0.0,0.0,0.0,2425.002,0.0,0.0,0.0,0.0,1.0
"""max""",7426.25,96.356667,93.666667,1440.333333,36792.684,13121.098822,8721.428435,17314.571469,4933.333333,1.0


In [45]:
describe_df(df, 140, 150)

statistic,peep_ordered_is_current_value,pao2_is_current_value,standardized_urinary_output,cumsum_in,cumsum_out,cumsum_balance
str,f64,f64,f64,f64,f64,f64
"""count""",27307.0,27307.0,23022.0,27307.0,27307.0,27307.0
"""null_count""",0.0,0.0,4285.0,0.0,0.0,0.0
"""mean""",0.499359,0.865675,28.708949,145.51852,1872.940523,-1727.422003
"""std""",0.500009,0.341007,22.229831,307.445419,1589.664218,1588.655758
"""min""",0.0,0.0,0.0,0.0,0.0,-36680.960625
"""25%""",0.0,1.0,14.738,1.0,935.714,-2295.044424
"""50%""",0.0,1.0,24.1706,30.530973,1565.742,-1444.998
"""75%""",1.0,1.0,37.4309,137.0,2425.002,-803.364667
"""max""",1.0,1.0,416.5586,5207.333333,36792.684,4746.633333


## Cohort Splilt

In [46]:
# 全施設を8:2にsplitする方針
df_development = df.filter(pl.col('hospital_id').is_in([1, 2, 3, 5, 6, 7, 8, 9, 10, 11]))
del df # メモリ節約

In [47]:
# record数と症例数を確認
print('Number of records (model development cohort):', len(df_development))
print('Number of icu_stay_id (model development cohort):', len(df_development['icu_stay_id'].unique()))
print('Average number of records per icu_stay_id (model development cohort):', len(df_development) / len(df_development['icu_stay_id'].unique()))

Number of records (model development cohort): 27307
Number of icu_stay_id (model development cohort): 27307
Average number of records per icu_stay_id (model development cohort): 1.0


In [48]:
df_development.write_csv(os.path.join(data_dir, df_development_file_name))